# RES-138 Stage B - Google Colab A100-80GB TEI operator

**This notebook orchestrates; it does not implement.** Everything load-bearing already
exists in the repository at `CODE_SHA`: the Stage-B plan, the TEI server-info contract,
the deployment floor, the preflight manifest and the GPU operator script itself
(`notebooks/res138_stage_b_gpu.py`). The notebook mounts Drive, clones the exact commit,
builds the pinned TEI release from its pinned source commit, launches it as a local
process on `127.0.0.1:8080`, proves its identity through `GET /info`, verifies the sealed
Stage-A bundle and the deterministic Stage-B plan, and then invokes the existing operator.

**Managed Colab is the GPU host.** No container runtime is required or used: TEI is
built from source and run as a local process. The benchmark contract refuses any TEI
endpoint that is not local to this host, so the server binds `127.0.0.1` only - no
tunnel, no proxy, no public port.

**Two modes.** `RUN_MODE = "preflight"` (the default) re-embeds only the frozen Stage-A
calibration set at both dimensions and writes `metrics = null`; it never touches the
production corpus and hard-stops at the end. `RUN_MODE = "full"` is reachable only with a
non-empty `APPROVED_PREFLIGHT_SHA256` naming a preflight an operator verified on the
workstation; the existing GPU operator remains the authority for every deeper
authorization check.

**If Colab disconnects after a preflight, reconnecting is allowed.** Re-run this notebook
from the top on the new runtime: it reinstalls the exact repository and the exact TEI
build, and full mode proceeds only if the existing contract identity checks still match
(`/info`, the Stage-B plan, the approved preflight digest and the byte-identical
calibration vectors). Do not edit evidence files on Drive; the byte-identical calibration
check is deliberately not weakened for reconnects. If the serving function changed, full
mode must stop.

**Secrets.** Nothing here needs a token: Qwen3-Embedding-0.6B is public. If a credential
is ever required, supply it through Colab userdata/secrets or environment variables and
never print it. The endpoint is never exposed publicly.

## 0. Operator parameters

In [ ]:
# --- Operator parameters. This cell is configuration only; nothing here executes. ---
#
# RUN_MODE may only be:
#   "preflight" -> calibration/equivalence evidence only; hard stop before any corpus pass
#   "full"      -> production corpus measurement, after local verification of the preflight
#
# FULL requires APPROVED_PREFLIGHT_SHA256 != "" and equal to the digest of the
# gpu-preflight.json an operator verified on the workstation. Preflight is the default.

REPO_URL = "https://github.com/Litju/DynamisRAG.git"
CODE_SHA = "f35487f4e9c5b36da7e0f9951698810f7107a985"

RUN_MODE = "preflight"
APPROVED_PREFLIGHT_SHA256 = ""

# The production dtype is declared before launch and observed from /info; it is never
# changed after seeing any equivalence output.
EXPECTED_PRECISION = "bfloat16"

TEI_REPO_URL = "https://github.com/huggingface/text-embeddings-inference.git"
TEI_COMMIT = "e80ef225ed0e6cb1717ce632a6a84b6cf211bb67"
TEI_VERSION = "1.9.4"

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
MODEL_REVISION = "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"

TEI_URL = "http://127.0.0.1:8080"

# The TEI 1.9.4 server's client batch ceiling. It must be at least the Stage-B plan's
# frozen document client batch size (8); 32 is the upstream default and is declared here
# so the launch and the /info verification read the same value.
TEI_MAX_CLIENT_BATCH_SIZE = 32

# Bounded startup wait: the first launch downloads the pinned Qwen weights and warms up.
HEALTH_TIMEOUT_SECONDS = 1200

# Drive layout. The sealed Stage A run lives under <DRIVE_ROOT>/runs/ and is discovered
# and verified by the repository loader, never assumed by name. STAGE_A_RUN may hold an
# explicit path to override discovery; the resolved absolute path is always printed.
DRIVE_ROOT = "/content/drive/MyDrive/DynamisRAG/RES-138"
DRIVE_RUNS_SUBDIR = "runs"
DRIVE_SOURCES_SUBDIR = "sources/beir"
STAGE_B_SUBDIR = "stage-b"
STAGE_A_RUN = ""

# All active work happens on the ephemeral disk; Drive holds verified archives and
# finished evidence only.
SCRATCH_ROOT = "/content/res138-stage-b"
REPO_DIR_NAME = "DynamisRAG"
TEI_DIR_NAME = "text-embeddings-inference"

## 1. Mount Google Drive and resolve the storage layout

Paths, not folder ids: the repository records the folder ids for provenance, and a run
that needed one to start would break the first time the tree was reorganised.

In [ ]:
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_RUNS_PATH = f"{DRIVE_ROOT}/{DRIVE_RUNS_SUBDIR}"
DRIVE_SOURCES_PATH = f"{DRIVE_ROOT}/{DRIVE_SOURCES_SUBDIR}"
STAGE_B_ROOT = f"{DRIVE_ROOT}/{STAGE_B_SUBDIR}"

SCRATCH = Path(SCRATCH_ROOT)
REPO_DIR = SCRATCH / REPO_DIR_NAME
TEI_DIR = SCRATCH / TEI_DIR_NAME
BEIR_CACHE = Path(DRIVE_SOURCES_PATH)
TEI_LOG = SCRATCH / "tei.log"
BUILD_LOG = SCRATCH / "tei-build.log"

SCRATCH.mkdir(parents=True, exist_ok=True)

missing = [
    path for path in (DRIVE_ROOT, DRIVE_RUNS_PATH, DRIVE_SOURCES_PATH) if not Path(path).is_dir()
]
if missing:
    raise RuntimeError(
        f"these Drive folders do not exist: {missing}. Create them once, or check that "
        f"DRIVE_ROOT is {DRIVE_ROOT!r}."
    )

Path(STAGE_B_ROOT).mkdir(parents=True, exist_ok=True)

print(f"scratch        {SCRATCH}")
print(f"repository     {REPO_DIR}")
print(f"tei source     {TEI_DIR}")
print(f"sealed runs    {DRIVE_RUNS_PATH}")
print(f"beir cache     {BEIR_CACHE}")
print(f"stage-b root   {STAGE_B_ROOT}")

## 2. Clean-runtime guard and GPU inspection

Raw `nvidia-smi` and the standard library only: the repository is not on this runtime
yet. The guard prints the full `nvidia-smi` output, requires exactly one usable GPU, and
refuses anything below the A100-80GB deployment floor before any model download or TEI
build. The repository's own `read_gpu_identity()` and `require_deployment_floor()` run
again after the checkout and must agree with the record captured here.

The notebook is intended for an A100 80GB. A T4, L4, A100 40GB or any other below-floor
device is refused explicitly and the run stops.

In [ ]:
import subprocess

try:
    probe = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
except OSError as error:
    raise RuntimeError(
        "nvidia-smi could not be executed. This notebook runs the TEI production server on a "
        "Colab GPU host and refuses any runtime without one."
    ) from error
if probe.returncode != 0:
    raise RuntimeError(f"nvidia-smi failed ({probe.returncode}): {probe.stderr.strip()[:400]}")
print(probe.stdout)

identity = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,uuid,compute_cap,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
    check=False,
)
if identity.returncode != 0:
    raise RuntimeError(
        f"the GPU identity query failed ({identity.returncode}): {identity.stderr.strip()[:400]}"
    )

lines = [line.strip() for line in identity.stdout.splitlines() if line.strip()]
if len(lines) != 1:
    raise RuntimeError(
        f"nvidia-smi reported {len(lines)} GPUs where this lane validates exactly one dedicated "
        "benchmark GPU. Select a single-GPU runtime."
    )
fields = [field.strip() for field in lines[0].split(",")]
if len(fields) != 5:
    raise RuntimeError(f"unexpected nvidia-smi identity fields: {fields!r}")
name, uuid, capability_text, memory_text, driver = fields

capability_parts = capability_text.split(".")
if len(capability_parts) != 2 or any(not part.isdigit() for part in capability_parts):
    raise RuntimeError(f"nvidia-smi reported compute capability {capability_text!r}")
if not memory_text.isdigit() or int(memory_text) < 1:
    raise RuntimeError(f"nvidia-smi reported total memory {memory_text!r}")

# The same five observations the repository's read_gpu_identity() returns. The later cell
# re-reads them through the repository and refuses if this record changed in any way.
EARLY_GPU = {
    "name": name,
    "uuid": uuid,
    "compute_capability": [int(capability_parts[0]), int(capability_parts[1])],
    "total_memory_bytes": int(memory_text) * 1024 * 1024,
    "driver_version": driver,
}

print()
print(f"gpu            {name}")
print(f"capability     {capability_parts[0]}.{capability_parts[1]}")
print(f"memory bytes   {EARLY_GPU['total_memory_bytes']}")
print(f"driver         {driver}")

if (
    tuple(EARLY_GPU["compute_capability"]) < (8, 0)
    or EARLY_GPU["total_memory_bytes"] < 80_000_000_000
):
    raise RuntimeError(
        "this runtime is below the RES-138 Stage-B deployment floor (compute capability >= 8.0 "
        f"and GPU memory >= 80_000_000_000 bytes); observed {name} at {capability_text} with "
        f"{EARLY_GPU['total_memory_bytes']} bytes. T4, L4 and A100-40GB runtimes are refused: "
        "select Runtime > Change runtime type > A100 GPU (80GB) and run this notebook from the "
        "top."
    )

## 3. Clone DynamisRAG and check out the exact `CODE_SHA`

GitHub is the only code transport. On a fresh runtime the repository is cloned with
`--no-checkout`; when it already exists - a second `Run all` in the same runtime - the
existing checkout is proven reusable (a Git worktree whose configured `origin` is exactly
`REPO_URL`, and clean) and then re-detached at exactly `CODE_SHA`. A foreign repository,
a non-repository path or local modifications are each refused and left untouched;
nothing is deleted, reset or cleaned.

In [ ]:
import subprocess
from pathlib import Path


def git(*arguments: str, cwd: Path | None = None) -> str:
    """Run one git command, refusing a non-zero exit rather than printing its noise."""
    completed = subprocess.run(
        ["git", *arguments],
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(arguments)} failed ({completed.returncode}): "
            f"{completed.stderr.strip()[:400]}"
        )
    return completed.stdout.strip()


def require_reusable_checkout(repo: Path, *, expected_origin: str) -> None:
    """Refuse a path that exists and is not a clean clone of the expected repository.

    Nothing here deletes, resets or cleans: a checkout that cannot be proven reusable is
    refused rather than replaced.
    """
    if not repo.is_dir():
        raise RuntimeError(f"{repo} exists and is not a directory. Refusing to replace it.")
    if git("rev-parse", "--is-inside-work-tree", cwd=repo) != "true":
        raise RuntimeError(f"{repo} is not a Git worktree. Refusing a foreign repository.")
    origin = git("remote", "get-url", "origin", cwd=repo)
    if origin != expected_origin:
        raise RuntimeError(
            f"{repo} has origin {origin!r} but this notebook clones {expected_origin!r}. "
            "Refusing a foreign repository."
        )
    dirty = git("status", "--porcelain", cwd=repo)
    if dirty:
        raise RuntimeError(
            f"the existing checkout at {repo} is not clean:\n{dirty[:500]}\n"
            "Refusing to change revisions over local modifications; nothing was reset or cleaned."
        )


def exact_checkout(repo: Path, *, origin: str, commit: str) -> str:
    """Clone or reuse ``repo`` and leave it clean and detached at exactly ``commit``."""
    if repo.exists():
        require_reusable_checkout(repo, expected_origin=origin)
        print(f"reusing the verified checkout at {repo}")
    else:
        git("clone", "--no-checkout", origin, str(repo))
    git("fetch", "--depth", "1", "origin", commit, cwd=repo)
    git("checkout", "--detach", commit, cwd=repo)
    head = git("rev-parse", "HEAD", cwd=repo)
    if head != commit:
        raise RuntimeError(f"checked out {head!r} but the pinned commit is {commit!r}")
    dirty = git("status", "--porcelain", cwd=repo)
    if dirty:
        raise RuntimeError(f"the checkout at {repo} is not clean:\n{dirty[:500]}")
    return head


REPO_HEAD = exact_checkout(REPO_DIR, origin=REPO_URL, commit=CODE_SHA)

print(f"repository     {REPO_DIR}")
print(f"HEAD           {REPO_HEAD}")
print(f"python         {sys.version.split()[0]}")

## 4. Install only what this lane needs, then import the exact checkout

The repository's dependency model is inspected from the checked-out `pyproject.toml`
first. Stage B needs the runtime dependencies (which include `httpx2`) and the NumPy that
Colab already provides; it does **not** need the Stage-A model stack in
`requirements/res138-colab.txt`, because TEI serves the model as a separate process.
Colab owns NumPy, and the install must not move it: a replacement under already-imported
modules is exactly the failure this check exists to catch.

In [ ]:
import importlib.metadata
import subprocess
import sys
import tomllib
from pathlib import Path

if tuple(sys.version_info[:2]) != (3, 12):
    raise RuntimeError(
        f"the checkout declares requires-python == 3.12.* and this runtime is "
        f"{sys.version.split()[0]}; pip would refuse that install or resolve around it."
    )


def installed_version(name: str) -> str:
    """The installed version of one distribution, or a marker if it is absent."""
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not-installed"


with (REPO_DIR / "pyproject.toml").open("rb") as handle:
    project = tomllib.load(handle)
declared = [str(requirement) for requirement in project["project"]["dependencies"]]
benchmark_group = [str(requirement) for requirement in project["dependency-groups"]["benchmark"]]

print("the checked-out runtime declares:")
for requirement in declared:
    print(f"  {requirement}")
print(f"and the benchmark group carries {benchmark_group}")
print(
    "stage B installs the runtime dependencies only; the Stage A model stack "
    "(requirements/res138-colab.txt) is not part of this lane."
)

NUMPY_BEFORE = installed_version("numpy")
runtime_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", str(REPO_DIR)],
    capture_output=True,
    text=True,
    check=False,
)
if runtime_install.returncode != 0:
    raise RuntimeError(
        f"installing the checked-out DynamisRAG runtime dependencies failed "
        f"({runtime_install.returncode}): {runtime_install.stderr[-1500:]}"
    )
NUMPY_AFTER = installed_version("numpy")
if NUMPY_AFTER != NUMPY_BEFORE:
    raise RuntimeError(
        f"the install changed numpy from {NUMPY_BEFORE!r} to {NUMPY_AFTER!r}. Colab owns numpy; "
        "installing a replacement inside the live kernel mixes already-imported modules with "
        "newly installed files. Start a fresh runtime."
    )

sys.path.insert(0, str(REPO_DIR / "src"))
import dynamisrag  # noqa: E402

resolved = Path(dynamisrag.__file__).resolve()
if not resolved.is_relative_to(REPO_DIR.resolve()):
    raise RuntimeError(
        f"dynamisrag resolved to {resolved}, outside the freshly checked-out {REPO_DIR}. "
        "Refusing a second, unrelated copy."
    )
if installed_version("httpx2") == "not-installed":
    raise RuntimeError("the TEI client dependency httpx2 is not installed after the install.")

print(f"numpy          {NUMPY_BEFORE} (unchanged)")
print(f"dynamisrag     {resolved}")
print(f"HEAD           {REPO_HEAD}")

## 5. Re-prove the deployment floor through repository code

The clean-runtime guard above is deliberately a pre-checkout refusal. Once the exact
checkout is importable, the repository is the authority: `read_gpu_identity()` re-reads
`nvidia-smi` and `require_deployment_floor()` applies the frozen floor, and any
disagreement with the early record stops the run.

In [ ]:
from dynamisrag.benchmark.gpu_runtime import read_gpu_identity
from dynamisrag.benchmark.production import require_deployment_floor

repository_gpu = read_gpu_identity()
capability = repository_gpu["compute_capability"]
require_deployment_floor(
    capability=(capability[0], capability[1]),
    total_memory_bytes=repository_gpu["total_memory_bytes"],
    operation="res138_stage_b_colab",
)
if repository_gpu != EARLY_GPU:
    raise RuntimeError(
        "the GPU observed through repository code differs from the clean-runtime guard's record: "
        f"{repository_gpu!r} != {EARLY_GPU!r}. A runtime whose hardware changed mid-session is "
        "not the runtime this run bound itself to."
    )

print(
    f"deployment floor satisfied by {repository_gpu['name']} "
    f"(capability {capability[0]}.{capability[1]}, {repository_gpu['total_memory_bytes']} bytes)"
)
print(f"driver         {repository_gpu['driver_version']}")

## 6. Verify the sealed Stage A bundle from Drive

The notebook consumes the sealed Stage A result; it never regenerates it. The run
directory is discovered under `<DRIVE_ROOT>/runs/` and every candidate is verified through
the repository's `load_sealed_stage_a`, which checks the sealed bundle digest, the
full-run digest, the Stage-A code commit, the pinned model revisions, the input policy and
the shortlist. The resolved absolute path is printed. If no run verifies against the seal,
the notebook stops with a path error rather than reconstructing Stage A.

In [ ]:
from dynamisrag.benchmark.stage_a import load_sealed_stage_a

RUNS_ROOT = Path(DRIVE_RUNS_PATH)

if STAGE_A_RUN:
    candidates = [Path(STAGE_A_RUN)]
else:
    candidates = sorted(
        path
        for path in RUNS_ROOT.iterdir()
        if path.is_dir()
        and (path / "bundle-manifest.json").is_file()
        and (path / "full-run.json").is_file()
    )
if not candidates:
    raise RuntimeError(
        f"no sealed Stage A run was found under {RUNS_ROOT}. Stage B qualifies the sealed Stage A "
        "result and never regenerates it; restore the bundle or set STAGE_A_RUN to its exact "
        "directory."
    )

verified = []
failures = {}
for path in candidates:
    try:
        verified.append((path, load_sealed_stage_a(path)))
    except Exception as error:
        failures[path] = error

if len(verified) != 1:
    detail = "\n".join(
        f"  {path}: {type(error).__name__}: {error}" for path, error in failures.items()
    )
    if not verified:
        raise RuntimeError(
            f"no Stage A run under {RUNS_ROOT} verifies against the sealed Stage A identity. "
            "Stage B refuses to qualify anything else:\n" + detail
        )
    raise RuntimeError(
        f"{len(verified)} Stage A runs under {RUNS_ROOT} verify against the seal. Refusing to "
        "guess which one Stage B qualifies; set STAGE_A_RUN explicitly."
    )

STAGE_A_RUN_DIRECTORY, sealed = verified[0]

print(f"sealed run     {STAGE_A_RUN_DIRECTORY}")
print(f"stage-a code   {sealed.seal.code_sha}")
print(f"bundle sha256  {sealed.reference.bundle_sha256}")
print(f"full-run sha   {sealed.reference.full_run_sha256}")
for model_id, dimension in sealed.reference.candidates:
    print(f"shortlist      {model_id}@{dimension}")

## 7. Verify and cache the frozen BEIR sources

The existing deterministic acquisition/verifier checks each archive against its frozen
digest before it is used and caches the verified bytes under Drive. The notebook does not
reimplement loading; the GPU operator performs the extraction and loading itself from the
same verified cache. The frozen digests are read from the repository, never restated here.

In [ ]:
from dynamisrag.benchmark.beir import verify_and_cache_beir_sources
from dynamisrag.benchmark.contracts import RES138_BEIR_SOURCES, RES138_WORKLOAD_NAMES

verified_sources = verify_and_cache_beir_sources(
    scratch_dir=SCRATCH,
    cache_dir=BEIR_CACHE,
)

frozen_digests = {spec.workload: spec.sha256 for spec in RES138_BEIR_SOURCES}
observed_digests = {entry.spec.workload: entry.sha256 for entry in verified_sources}
if observed_digests != frozen_digests:
    raise RuntimeError(
        f"the verified BEIR archives are not the frozen ones: {observed_digests!r} != "
        f"{frozen_digests!r}"
    )
if tuple(observed_digests) != RES138_WORKLOAD_NAMES:
    raise RuntimeError("the verified BEIR workloads are not the frozen workload set")

for entry in verified_sources:
    print(f"{entry.spec.workload:<11} {entry.sha256}  {entry.path}")
print(f"beir cache     {BEIR_CACHE}")

## 8. Build and print the Stage-B plan identity

The plan is the deterministic, machine-independent identity of this run: the sealed
Stage-A reference, the pinned model revision, the two dimensions, the 8192/right semantic
boundary, the frozen client measurement policy, the equivalence gate and the deployment
floor. It is built from the verified sealed reference and `CODE_SHA` and nothing else, so
preflight and full compute the same digest - and the evidence directory is derived from
that digest, which is how both modes share one directory.

In [ ]:
from dynamisrag.benchmark.stage_b import build_stage_b_plan

plan = build_stage_b_plan(reference=sealed.reference, code_sha=CODE_SHA)
if plan.dimensions != (512, 1024):
    raise RuntimeError(f"the Stage-B plan dimensions are {plan.dimensions!r}, not (512, 1024)")
if plan.model_revision != MODEL_REVISION:
    raise RuntimeError(
        f"the plan serves revision {plan.model_revision!r}, not the pinned {MODEL_REVISION!r}"
    )
if plan.model_ids != (MODEL_ID,):
    raise RuntimeError(f"the plan shortlists {plan.model_ids!r}, not the sealed {MODEL_ID!r}")

plan_payload = plan.payload()
if plan_payload["code_sha"] != CODE_SHA:
    raise RuntimeError("the plan does not bind this run's CODE_SHA")

EVIDENCE_DIR = Path(STAGE_B_ROOT) / f"plan-{plan.sha256}"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

print(f"stage-b plan   {plan.sha256}")
print(f"model revision {plan.model_revision}")
print(f"dimensions     {plan.dimensions}")
print(f"semantic boundary   {plan_payload['semantic_boundary']}")
print(f"client policy  {plan.client_policy}")
print(f"equivalence gate    {plan.gate.payload()}")
print(f"deployment floor    {plan_payload['deployment_floor']}")
print(f"evidence dir   {EVIDENCE_DIR}")

## 9. Clone TEI 1.9.4 at the pinned source commit

The server is built from the immutable upstream commit, never from a moving branch or a
mutable installed binary. The checkout is detached at exactly `TEI_COMMIT`, the tree is
clean, and the source's own `rust-toolchain.toml` must declare the pinned Rust channel.

In [ ]:
TEI_HEAD = exact_checkout(TEI_DIR, origin=TEI_REPO_URL, commit=TEI_COMMIT)

source_toolchain = TEI_DIR / "rust-toolchain.toml"
if not source_toolchain.is_file():
    raise RuntimeError(
        f"{TEI_DIR} has no rust-toolchain.toml; the pinned TEI source declares Rust 1.92.0 and "
        "this notebook refuses to build a toolchain the source did not pin."
    )
with source_toolchain.open("rb") as handle:
    toolchain = tomllib.load(handle)
channel = toolchain.get("toolchain", {}).get("channel")
if channel != "1.92.0":
    raise RuntimeError(
        f"the TEI source at {TEI_HEAD} declares Rust {channel!r}, not the pinned 1.92.0. "
        "Refusing to build with a toolchain the source did not pin."
    )

print(f"tei source     {TEI_DIR}")
print(f"HEAD           {TEI_HEAD}")
print(f"toolchain      {channel}")

## 10. Provision Rust 1.92.0 and the TEI build dependencies

Rust is installed with `rustup` at the exact toolchain the source declares, and the
upstream-supported CUDA build path is selected for this Ampere-or-newer GPU. The CUDA
toolkit must be present: a CPU-only TEI is not a fallback, because a CPU server would
serve a different latency and the benchmark lane refuses it.

In [ ]:
import os
from pathlib import Path

for command in (
    ["apt-get", "update", "-qq"],
    ["apt-get", "install", "-y", "--no-install-recommends", "libssl-dev", "gcc", "pkg-config"],
):
    completed = subprocess.run(command, capture_output=True, text=True, check=False)
    if completed.returncode != 0:
        raise RuntimeError(
            f"{' '.join(command)} failed ({completed.returncode}): {completed.stderr[-1200:]}"
        )
print("build dependencies present: libssl-dev, gcc, pkg-config")

rustup_init = SCRATCH / "rustup-init.sh"
download = subprocess.run(
    [
        "curl",
        "--proto",
        "=https",
        "--tlsv1.2",
        "-sSf",
        "https://sh.rustup.rs",
        "-o",
        str(rustup_init),
    ],
    capture_output=True,
    text=True,
    check=False,
)
if download.returncode != 0:
    raise RuntimeError(f"downloading rustup failed ({download.returncode}): {download.stderr[-800:]}")
rust_install = subprocess.run(
    [
        "sh",
        str(rustup_init),
        "-y",
        "--default-toolchain",
        "1.92.0",
        "--profile",
        "minimal",
        "--component",
        "rustfmt",
        "--component",
        "clippy",
        "--no-modify-path",
    ],
    capture_output=True,
    text=True,
    check=False,
)
if rust_install.returncode != 0:
    raise RuntimeError(
        f"installing Rust 1.92.0 failed ({rust_install.returncode}): {rust_install.stderr[-1200:]}"
    )
os.environ["PATH"] = str(Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]

rustc_version = subprocess.run(["rustc", "--version"], capture_output=True, text=True, check=False)
cargo_version = subprocess.run(["cargo", "--version"], capture_output=True, text=True, check=False)
if rustc_version.returncode != 0 or "1.92.0" not in rustc_version.stdout:
    raise RuntimeError(f"rustc is not the pinned 1.92.0: {rustc_version.stdout.strip()!r}")
if cargo_version.returncode != 0:
    raise RuntimeError(f"cargo is not usable: {cargo_version.stderr.strip()[:400]}")
print(rustc_version.stdout.strip())
print(cargo_version.stdout.strip())

cuda_home = Path("/usr/local/cuda")
nvcc = cuda_home / "bin" / "nvcc"
if not nvcc.is_file():
    raise RuntimeError(
        "the Colab runtime has no CUDA toolkit at /usr/local/cuda (nvcc not found). The pinned "
        "TEI build compiles CUDA kernels and cannot fall back to CPU: a CPU TEI would serve the "
        "wrong latency and the benchmark lane refuses it. Select a GPU runtime with the CUDA "
        "toolkit, or stop and report this blocker rather than substituting another server."
    )
nvcc_version = subprocess.run([str(nvcc), "--version"], capture_output=True, text=True, check=False)
print(nvcc_version.stdout.strip().splitlines()[-1])
os.environ["CUDA_HOME"] = str(cuda_home)
os.environ["CUDA_PATH"] = str(cuda_home)
os.environ["PATH"] = f"{cuda_home / 'bin'}{os.pathsep}" + os.environ["PATH"]

# Build the CUDA kernels for the exact device this run verified. The upstream CUDA build
# path uses the same variable, and 80 (A100, capability 8.0) is the intended target.
gpu_capability = EARLY_GPU["compute_capability"]
os.environ["CUDA_COMPUTE_CAP"] = f"{gpu_capability[0]}{gpu_capability[1]}"
print(f"CUDA_COMPUTE_CAP {os.environ['CUDA_COMPUTE_CAP']}")

## 11. Build and install TEI with the CUDA router

The upstream-supported Ampere build path: `cargo install --path router -F candle-cuda`
from the exact source checkout. The output streams to the cell and to a log so a failure
is diagnosable. No compiled binary from another runtime is ever reused: the router is
rebuilt from the verified source tree in this session.

In [ ]:
build_command = ["cargo", "install", "--path", "router", "-F", "candle-cuda"]
print(f"building {' '.join(build_command)} from {TEI_DIR}; this takes a while")
with BUILD_LOG.open("w", encoding="utf-8") as log:
    build = subprocess.Popen(
        build_command,
        cwd=str(TEI_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy(),
    )
    if build.stdout is None:
        raise RuntimeError("the build process did not expose its output")
    for line in build.stdout:
        print(line, end="")
        log.write(line)
    returncode = build.wait()
if returncode != 0:
    tail = "\n".join(BUILD_LOG.read_text(encoding="utf-8").splitlines()[-80:])
    raise RuntimeError(f"cargo install failed ({returncode}). Tail of {BUILD_LOG}:\n{tail}")

router_binary = Path.home() / ".cargo" / "bin" / "text-embeddings-router"
if not router_binary.is_file():
    raise RuntimeError(f"cargo install did not produce {router_binary}")
print(f"installed      {router_binary}")

## 12. Verify the TEI executable and source identity before launch

The executable must be runnable, and it must have been built from the clean, exact source
checkout. `GET /info` after launch is the final authority on the server's identity; this
cell is the pre-launch proof that the binary was produced from the pinned tree.

In [ ]:
router_binary = Path.home() / ".cargo" / "bin" / "text-embeddings-router"

help_result = subprocess.run(
    [str(router_binary), "--help"], capture_output=True, text=True, check=False
)
if help_result.returncode != 0:
    raise RuntimeError(
        f"text-embeddings-router --help failed ({help_result.returncode}): "
        f"{help_result.stderr[-800:]}"
    )
version_result = subprocess.run(
    [str(router_binary), "--version"], capture_output=True, text=True, check=False
)
if version_result.returncode != 0:
    raise RuntimeError(
        f"text-embeddings-router --version failed ({version_result.returncode}): "
        f"{version_result.stderr[-800:]}"
    )
print(version_result.stdout.strip() or version_result.stderr.strip())

source_head = git("rev-parse", "HEAD", cwd=TEI_DIR)
source_dirty = git("status", "--porcelain", cwd=TEI_DIR)
if source_head != TEI_COMMIT or source_dirty:
    raise RuntimeError(
        f"the TEI source is at {source_head!r} with status {source_dirty!r}; the binary must be "
        f"built from the clean pinned {TEI_COMMIT}."
    )
if TEI_VERSION not in version_result.stdout:
    raise RuntimeError(
        f"the executable does not identify as TEI {TEI_VERSION}; /info would refuse it anyway"
    )
print(f"built from     {TEI_DIR} HEAD {source_head} (clean)")

## 13. Launch TEI in the background on the frozen server configuration

Any stale local router process is terminated first, and the port must be free before the
new server starts. The server log is written to the scratch disk and printed as a path.
The command uses exactly the frozen Stage-B configuration: the pinned model at the pinned
revision, the declared production dtype, the 8192-token boundary with auto-truncation, the
declared client batch ceiling, and `127.0.0.1:8080` only. TEI 1.9.4 has no
`--max-input-length` router option and none is passed.

In [ ]:
import socket
import time
from urllib.parse import urlsplit

from dynamisrag.benchmark.contracts import RES138_INPUT_MAX_TOKENS

tei_endpoint = urlsplit(TEI_URL)
if tei_endpoint.hostname not in ("127.0.0.1", "localhost", "::1"):
    raise RuntimeError(
        f"TEI_URL {TEI_URL!r} is not local to the GPU operator host; the benchmark lane admits "
        "only a local endpoint."
    )
tei_host = "127.0.0.1" if tei_endpoint.hostname == "localhost" else tei_endpoint.hostname
tei_port = tei_endpoint.port or 80


def port_is_open(host: str, port: int) -> bool:
    """One local connect attempt; the launch refuses to start over an unknown listener."""
    family = socket.AF_INET6 if ":" in host else socket.AF_INET
    with socket.socket(family, socket.SOCK_STREAM) as probe:
        probe.settimeout(0.5)
        return probe.connect_ex((host, port)) == 0


subprocess.run(
    ["pkill", "-f", "text-embeddings-router"], capture_output=True, text=True, check=False
)
for _ in range(30):
    if not port_is_open(tei_host, tei_port):
        break
    time.sleep(1.0)
if port_is_open(tei_host, tei_port):
    raise RuntimeError(
        f"{tei_host}:{tei_port} is still occupied after terminating stale router processes. "
        "Refusing to launch over an unknown listener."
    )

if RES138_INPUT_MAX_TOKENS != 8192:
    raise RuntimeError(
        f"the Stage-B reference boundary is {RES138_INPUT_MAX_TOKENS}, not the frozen 8192-token "
        "boundary this startup is written against."
    )

tei_arguments = [
    str(router_binary),
    "--model-id",
    MODEL_ID,
    "--revision",
    MODEL_REVISION,
    "--dtype",
    EXPECTED_PRECISION,
    "--max-batch-tokens",
    str(RES138_INPUT_MAX_TOKENS),
    "--auto-truncate",
    "--max-client-batch-size",
    str(TEI_MAX_CLIENT_BATCH_SIZE),
    "--hostname",
    tei_host,
    "--port",
    str(tei_port),
]

tei_environment = os.environ.copy()
tei_environment["HF_HOME"] = str(SCRATCH / "hf-home")
Path(tei_environment["HF_HOME"]).mkdir(parents=True, exist_ok=True)

print(f"launching      {' '.join(tei_arguments)}")
print(f"server log     {TEI_LOG}")
tei_log_handle = TEI_LOG.open("w", encoding="utf-8")
tei_process = subprocess.Popen(
    tei_arguments,
    cwd=str(TEI_DIR),
    stdout=tei_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    env=tei_environment,
)
print(f"tei pid        {tei_process.pid}")

## 14. Wait for `/health` with a bounded timeout

`/health` proves liveness only. The loop is bounded, observes whether the process exited
during startup, and prints the tail of the server log when it fails.

In [ ]:
import urllib.error
import urllib.request

health_url = f"{TEI_URL.rstrip('/')}/health"


def server_log_tail(lines: int = 60) -> str:
    """The last lines of the server log, or a marker when there is none yet."""
    if not TEI_LOG.is_file():
        return "(no server log yet)"
    return "\n".join(TEI_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])


deadline = time.monotonic() + HEALTH_TIMEOUT_SECONDS
healthy = False
while time.monotonic() < deadline:
    if tei_process.poll() is not None:
        raise RuntimeError(
            f"the TEI server exited during startup with status {tei_process.returncode}. Tail of "
            f"{TEI_LOG}:\n{server_log_tail()}"
        )
    try:
        with urllib.request.urlopen(health_url, timeout=5.0) as response:
            if response.status == 200:
                healthy = True
                break
    except (urllib.error.URLError, TimeoutError, ConnectionError):
        pass
    time.sleep(5.0)

if not healthy:
    raise RuntimeError(
        f"TEI did not answer GET {health_url} within {HEALTH_TIMEOUT_SECONDS} seconds. Tail of "
        f"{TEI_LOG}:\n{server_log_tail()}"
    )
if tei_process.poll() is not None:
    raise RuntimeError(
        f"the TEI server exited immediately after answering /health with status "
        f"{tei_process.returncode}; refusing to record a dead server's identity"
    )
print(f"healthy        {health_url} (pid {tei_process.pid})")

## 15. Fetch and print `/info`

`/info` is the identity authority. Its JSON is printed verbatim so the observed record is
visible before anything interprets it.

In [ ]:
import json
import urllib.request

info_url = f"{TEI_URL.rstrip('/')}/info"
with urllib.request.urlopen(info_url, timeout=30.0) as response:
    info_payload = json.loads(response.read().decode("utf-8"))
print(json.dumps(info_payload, indent=2, sort_keys=True))

## 16. Verify `/info` through the repository contract

`parse_tei_server_info` is the repository's canonical validation of the serving identity:
the frozen TEI version, the plan's model and pinned revision, the declared production
dtype, the 8192/8192 boundary with auto-truncation, and the client batch ceiling. The
serving identity digest is computed by the repository class and printed; the notebook
never reimplements it.

In [ ]:
from dynamisrag.benchmark.contracts import RES138_MODEL_CANDIDATES
from dynamisrag.benchmark.tei_server import parse_tei_server_info

if not info_payload.get("sha"):
    raise RuntimeError(
        "the locally built TEI /info did not report a build sha. The pinned source embeds it "
        "from the checkout's git HEAD; an absent sha means the binary was not built from the "
        "verified source tree. Stop and report the blocker instead of substituting a server."
    )

candidate = next(entry for entry in RES138_MODEL_CANDIDATES if entry.model_id == plan.model_ids[0])
server_info = parse_tei_server_info(
    info_payload,
    expected_model_id=candidate.model_id,
    expected_model_revision=plan.model_revision,
    expected_precision=EXPECTED_PRECISION,
    min_max_client_batch_size=plan.document_client_batch_size,
    operation="res138_stage_b_colab",
)

TEI_SERVER_SHA256 = server_info.sha256
print(f"tei_server_sha256      {TEI_SERVER_SHA256}")
print(f"version                {server_info.version}")
print(f"model_id               {server_info.model_id}")
print(f"model_sha              {server_info.model_sha}")
print(f"model_dtype            {server_info.model_dtype}")
print(f"max_input_length       {server_info.max_input_length}")
print(f"max_batch_tokens       {server_info.max_batch_tokens}")
print(f"auto_truncate          {server_info.auto_truncate}")
print(f"max_client_batch_size  {server_info.max_client_batch_size}")
print(f"sha                    {server_info.sha}")
print(f"docker_label           {server_info.docker_label}")
print(f"max_concurrent_requests {server_info.max_concurrent_requests}")
print(f"max_batch_requests     {server_info.max_batch_requests}")
print(f"tokenization_workers   {server_info.tokenization_workers}")

## 17. Execute PRELIGHT or FULL by invoking the existing GPU operator

The notebook does not reproduce the operator. It builds the argument vector
programmatically - no shell-string interpolation, so nothing is lost to quoting - and
invokes `notebooks/res138_stage_b_gpu.py` from the exact checkout. The notebook performs
only a cheap early refusal for full mode; the operator is the authority for every deeper
authorization check. Preflight and full use the same `CODE_SHA`, Stage-A bundle, model
revision, precision, TEI identity and evidence directory, and full never creates a fresh
preflight.

In [ ]:
import re

from dynamisrag.benchmark.gpu_preflight import RES138_GPU_PREFLIGHT_FILENAME

if RUN_MODE not in ("preflight", "full"):
    raise RuntimeError(f"RUN_MODE is {RUN_MODE!r}; RUN_MODE may only be 'preflight' or 'full'.")
if RUN_MODE == "full":
    if re.fullmatch(r"[0-9a-f]{64}", APPROVED_PREFLIGHT_SHA256) is None:
        raise RuntimeError(
            "full mode requires APPROVED_PREFLIGHT_SHA256 to be the 64-character lowercase "
            "SHA-256 of the gpu-preflight.json an operator verified locally. Re-run in preflight "
            "mode first and return its digest."
        )
    if not (EVIDENCE_DIR / RES138_GPU_PREFLIGHT_FILENAME).is_file():
        raise RuntimeError(
            f"full mode requires the approved preflight at "
            f"{EVIDENCE_DIR / RES138_GPU_PREFLIGHT_FILENAME}. The corpus is measured only after "
            "the approved preflight exists in the same evidence directory."
        )

operator_command = [
    sys.executable,
    str(REPO_DIR / "notebooks" / "res138_stage_b_gpu.py"),
    "--mode",
    RUN_MODE,
    "--bundle",
    str(STAGE_A_RUN_DIRECTORY),
    "--beir-cache",
    str(BEIR_CACHE),
    "--scratch",
    str(SCRATCH),
    "--code-sha",
    CODE_SHA,
    "--tei-url",
    TEI_URL,
    "--expected-precision",
    EXPECTED_PRECISION,
    "--out",
    str(EVIDENCE_DIR),
]
if RUN_MODE == "full":
    operator_command += ["--approved-preflight-sha256", APPROVED_PREFLIGHT_SHA256]

operator_environment = os.environ.copy()
operator_environment["PYTHONPATH"] = str(REPO_DIR / "src")

print("running:")
print("  " + " ".join(operator_command))
completed = subprocess.run(
    operator_command,
    cwd=str(REPO_DIR),
    env=operator_environment,
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout, end="")
if completed.stderr:
    print(completed.stderr, end="")
if completed.returncode != 0:
    raise RuntimeError(f"the Stage B GPU operator exited with status {completed.returncode}")
print(f"operator exited 0; evidence is under {EVIDENCE_DIR}")

## 18. Artifact inspection and SHA printing

The preflight manifest is re-read through the repository, every file in the evidence
directory is listed with its SHA-256, and the expected artifact set is derived from the
manifest records and the plan dimensions - never hand-typed. Preflight must not have
produced a full-corpus artifact.

In [ ]:
import shutil

from dynamisrag.benchmark.artifacts import file_sha256
from dynamisrag.benchmark.gpu_evidence import full_evidence_filename
from dynamisrag.benchmark.gpu_preflight import read_gpu_preflight

shutil.copyfile(TEI_LOG, EVIDENCE_DIR / "tei.log")

manifest = read_gpu_preflight(EVIDENCE_DIR / RES138_GPU_PREFLIGHT_FILENAME)
print(f"preflight sha  {manifest.sha256}")
print(f"plan sha       {manifest.stage_b_plan_sha256}")
print(f"server sha     {manifest.tei_server_sha256}")
print(f"precision      {manifest.precision}")

required = {RES138_GPU_PREFLIGHT_FILENAME}
for dimension in plan.dimensions:
    record = manifest.record_for(dimension)
    required.add(record.evidence_file)
    required.add(record.vector_file)
if RUN_MODE == "preflight":
    for dimension in plan.dimensions:
        full_name = full_evidence_filename(dimension)
        if (EVIDENCE_DIR / full_name).exists():
            raise RuntimeError(
                f"{full_name} exists after a preflight run. Preflight must produce calibration "
                "evidence only; refusing to continue."
            )
if RUN_MODE == "full":
    for dimension in plan.dimensions:
        required.add(full_evidence_filename(dimension))

present = sorted(path.name for path in EVIDENCE_DIR.iterdir() if path.is_file())
missing = sorted(required - set(present))
if missing:
    raise RuntimeError(f"the evidence directory is missing {missing}; found {present}")

print()
print(f"evidence directory  {EVIDENCE_DIR}")
for name in present:
    print(f"{file_sha256(EVIDENCE_DIR / name)}  {name}")

print()
if RUN_MODE == "preflight":
    print("Next: return every artifact above to the local workstation and verify BOTH dimensions:")
    for dimension in plan.dimensions:
        evidence_file = manifest.record_for(dimension).evidence_file
        print(
            "  uv run dynamisrag benchmark verify-gpu-evidence "
            f"--bundle {STAGE_A_RUN_DIRECTORY} --code-sha {CODE_SHA} "
            f"--evidence {EVIDENCE_DIR / evidence_file}"
        )
    print(f"  APPROVED_PREFLIGHT_SHA256 = {manifest.sha256!r}")
    print("  Only after both dimensions pass, set RUN_MODE = 'full' and re-run this notebook.")
else:
    print("Full evidence is written. Local verification and qualification assembly decide what it")
    print("means; this notebook selects no dimension and configures no production default.")

## 19. Hard stop / completion

No path below a preflight run exists. Nothing is selected and no production default is
bound.

In [ ]:
if RUN_MODE == "preflight":
    print()
    print("STAGE-B GPU PREFLIGHT COMPLETE")
    print("NO PRODUCTION METRICS WERE MEASURED")
    print("NO DEFAULT WAS SELECTED")
    print()
    print("Return these artifacts to the local workstation and verify both 512 and 1024")
    print("before authorizing full mode.")
    raise SystemExit("Preflight hard stop: the full corpus path is not reachable in this mode.")

print()
print("STAGE-B GPU FULL EVIDENCE COMPLETE")
print("NO PRODUCTION DEFAULT WAS SELECTED; qualification assembly and selection are local steps.")
raise SystemExit("Full evidence written; this notebook stops here.")